# Tests / Experiments
This notebook mostly contains some visualization / experiments on unrelated things in the `headset_localization` package.

In [ ]:
%load_ext autoreload
%autoreload 2
print(__debug__)

import logging
logging.basicConfig(level=logging.INFO)

import seaborn as sns
sns.set_theme(style="whitegrid", context="paper", font_scale=1)


import numpy as np
import random, torch, os, cv2

seed = 1

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
cv2.setRNGSeed(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["PYTHONHASHSEED"] = str(seed)

import matplotlib.pyplot as plt

from headset_localization import *

In [ ]:
from shared.complete_robot_scan import CompleteRobotScan
robot_data_folder_location = "../example_datasets/example_small_aruco1"
vrs_file_location = "../example_datasets/small_aruco1_sitting_20fps.vrs"

#robot_data_folder_location = "../example_datasets/small_aruco_2"
#vrs_file_location = "../example_datasets/small_aruco_2_2.vrs"


robot_data = CompleteRobotScan.from_folder(robot_data_folder_location)
robot_env = Scanned3dEnvironment.from_gathered_robot_data(
        robot_data = robot_data,
        number_of_sampled_datapoints=10,
        sample_datapoints_based_on_aruco_corectness = False,
        only_sample_robot_datapoints_w_marker_estimates = True,
        markers_use_advanced_removal=True,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.05, use_depth_images_if_provided=True),
        est3d_xyz_icp_config=ICPAlignmentConfig(do_alginment=False)
)

labeled_headset_data = bind_headset_recording_to_scan(
        headset_data = HeadsetRecording.from_vrs_file(vrs_file_location),
        robot_data = robot_data
)

visualize_loaded_data = False

if visualize_loaded_data:
    visualize_robot_camera_environment_combo(robot_env=robot_env, headset_data=labeled_headset_data)


In [ ]:
#pne_optimizer = PyposePNEOptimizer(PyposePnEOptimizerConfig())
#pne_optimizer = PnEDeltaPoseLBFGSOptimizer(time_tracker=tt_pne)

predictor = EllipsoidLocalizer(
        cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
        cam1_bgr_images=robot_env.robot_bgr_images,
        cam1_xyz_images=robot_env.robot_xyz_images,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            rotation_augmentations=[Rotate180Deg],
            extract_and_match=ExtractAndLightGlue(),
            ransac_config=pose_estimation_ransaac_config_less_precise,
        ),
        pne_optimizer=PnEDeltaPoseAdamOptimizer(),
        ellipsoid_refinement_at_res=(1400, 1400),
        cam1_segmenter=SAM3Segmenter(Sam3Prompt()),
        cam2_segmenter=YOLOv26Segmenter("yoloe-26l-seg.pt"),
        matching_config=GaussianMatchingConfig(dummy_value=0.01),
        visualize_pne_optimisation=False,
        visualize_environment_generation=True
)

In [ ]:
ge = FastRayIntersectionError(points=robot_env.robot_xyz_images.reshape(-1,3), intrinsics=labeled_headset_data.intrinsic_cam_mtx, visualize=True)

grade = PredictionOnDataset(
    predictor=predictor,
    headset_data=labeled_headset_data
)

i, pred, label = grade.comparable_poses[0]

pixels = sample_pixel_neighborhood(center=(550,700))

fig, ax = plt.subplots(1,1)
ax.imshow(labeled_headset_data.bgr_image_s[i])
ax.scatter(pixels[:, 0], pixels[:,1], c='red', marker='o', s=2, label='Type A')

print(ge.calculate_gripping_differences_4_pixels(base_t_cam_s=np.array([pred, label]), pixels_batch=np.array([pixels, pixels])))

### Ellipsoid fitting

In [ ]:
import open3d as o3d

simple_fitter = SimpleEllipsoidFitter(visualize=False)
mvee_fitter = MVEEEllipsoidFitter(visualize=False, contamination=0.0)
ls_fitter = LeastShellDistanceEllipsoidFitter(visualize=False, size_penalty=0.95)

fitters = [simple_fitter, mvee_fitter, ls_fitter]

n_samples = 1000

points_x = np.random.rand(n_samples)-0.5
points_y = np.random.rand(n_samples)-0.5
points_z = -points_x**2 - points_y**2

pc = np.column_stack([points_x, points_y, points_z]) + 0.01 * np.random.rand(n_samples, 3)
print(f"pc: {pc.shape}")

for fitter in fitters:
    base_t_ellipsoid, primal_quadratic = fitter.fit_ellipsoid(points=pc)
    ls = create_ellipsoid_lineset(base_t_ellipsoid, primal_quadratic)

    pcd1 = o3d.geometry.PointCloud()
    pcd1.points = o3d.utility.Vector3dVector(pc)
    pcd1.paint_uniform_color([1,0,0])

    frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.05)
    frame.transform(base_t_ellipsoid)

    to_vis = [ls, pcd1, frame]
    o3d.visualization.draw_geometries(to_vis, f"Ellipsoid fit visualization", 1200, 1200)

In [ ]:
import open3d as o3d

simple_fitter = SimpleEllipsoidFitter(visualize=False)
mvee_fitter = MVEEEllipsoidFitter(visualize=False, contamination=0.0)
ls_fitter = LeastShellDistanceEllipsoidFitter(visualize=False, size_penalty=0.95)

fitters = [simple_fitter, mvee_fitter, ls_fitter]

n_samples = 1000

points_x = np.random.rand(n_samples)-0.5
points_y = np.random.rand(n_samples)-0.5
points_z = -points_x**2 - points_y**2

pc = np.column_stack([points_x, points_y, points_z]) + 0.01 * np.random.rand(n_samples, 3)
print(f"pc: {pc.shape}")

for fitter in fitters:
    base_t_ellipsoid, primal_quadratic = fitter.fit_ellipsoid(points=pc)
    ls = create_ellipsoid_lineset(base_t_ellipsoid, primal_quadratic)

    pcd1 = o3d.geometry.PointCloud()
    pcd1.points = o3d.utility.Vector3dVector(pc)
    pcd1.paint_uniform_color([1,0,0])

    frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.05)
    frame.transform(base_t_ellipsoid)

    to_vis = [ls, pcd1, frame]
    o3d.visualization.draw_geometries(to_vis, f"Ellipsoid fit visualization", 1200, 1200)

In [ ]:
from shared.complete_robot_scan import CompleteRobotScan
robot_data_folder_location = "../example_datasets/example_small_aruco1"
vrs_file_location = "../example_datasets/small_aruco1_sitting_20fps.vrs"


robot_data = CompleteRobotScan.from_folder(robot_data_folder_location)
n1 = 4
n = 6

robot_data_red = CompleteRobotScan(
    name=robot_data.name, 
    bgr_images=robot_data.bgr_images[n1:n],
    depth_images=robot_data.depth_images[n1:n],
    cam_intrinsic_mtx=robot_data.cam_intrinsic_mtx, 
    base_t_gripper_s=robot_data.base_t_gripper_s[n1:n],
    camera_t_marker_s=robot_data.camera_t_marker_s[n1:n], 
    marker_detector=robot_data.marker_detector, 
    gripper_t_cam=robot_data.gripper_t_cam
)


robot_env_red = Scanned3dEnvironment.from_gathered_robot_data(
        robot_data = robot_data_red,
        number_of_sampled_datapoints=n,
        sample_datapoints_based_on_aruco_corectness = False,
        only_sample_robot_datapoints_w_marker_estimates = False,
        markers_use_advanced_removal=True,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.05, use_depth_images_if_provided=True),
        est3d_xyz_icp_config=ICPAlignmentConfig(do_alginment=False)
)


img_rgb = cv2.cvtColor(robot_data_red.bgr_images[0], cv2.COLOR_BGR2RGB)
plt.imshow(img_rgb)
plt.axis('off')
plt.tight_layout(pad=0)
plt.show()

depth = robot_data_red.depth_images[0]
depth_normalized = (depth - depth.min()) / (depth.max() - depth.min())
plt.imshow(depth_normalized, cmap='gray')
plt.axis('off')
plt.tight_layout(pad=0)
plt.show()


img_rgb_no_marker = cv2.cvtColor(robot_env_red._robot_bgr_images[0], cv2.COLOR_BGR2RGB)
plt.imshow(img_rgb_no_marker)
plt.axis('off')
plt.tight_layout(pad=0)
plt.show()


robot_data_red.save("../img_scan", new_name="test")
robot_env_red.save("../img_env", new_name="test2")


world_points = robot_env_red.robot_xyz_images[0].reshape(-1,3)
world_colors = robot_env_red.robot_bgr_images[0].reshape(-1,3).astype(np.float32)[:, ::-1]/255
no_nan_mask = np.isfinite(world_points).all(axis=-1)

import open3d as o3d
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(world_points[no_nan_mask])
pcd.colors = o3d.utility.Vector3dVector(world_colors[no_nan_mask])
o3d.visualization.draw_geometries([pcd], f"Robot environment 1 image visualization")


labeled_headset_data = bind_headset_recording_to_scan(
        headset_data = HeadsetRecording.from_vrs_file(vrs_file_location),
        robot_data = robot_data
)

headset_img = cv2.cvtColor(labeled_headset_data.bgr_image_s[30], cv2.COLOR_BGR2RGB)
plt.imshow(headset_img)
plt.axis('off')
plt.tight_layout(pad=0)
plt.show()


In [ ]:
el = EllipsoidLocalizer(
    cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
    cam1_bgr_images=robot_env.robot_bgr_images,
    cam1_xyz_images=robot_env.robot_xyz_images, 
    cam1_segmenter=SAM3Segmenter(Sam3Prompt()),
    ellipsoid_fitter=LeastShellDistanceEllipsoidFitter(),
    visualize_environment_generation=True,
    visualize_segmentation_masks=True,
    visualize_pne_optimisation=True
)


visualize_primal_quadratics(
    base_t_ellipsoid_s=el.base_t_ellipsoid_s,
    primal_quadratic_s=el.primal_quadratic_s,
    avg_colors=plt.colormaps['viridis'](np.linspace(0, 1, el.base_t_ellipsoid_s.shape[0]))[:, :3]
)

el.est_base_t_cam2(cam2_bgr_image=labeled_headset_data.bgr_image_s[30])

In [ ]:
robot_data_folder_location_2 = "../example_datasets/small_aruco_2"
vrs_file_location_2 = "../example_datasets/small_aruco_2_2.vrs"


robot_data_2 = CompleteRobotScan.from_folder(robot_data_folder_location)
robot_env_2 = Scanned3dEnvironment.from_gathered_robot_data(
        robot_data = robot_data_2,
        number_of_sampled_datapoints=10,
        sample_datapoints_based_on_aruco_corectness = False,
        only_sample_robot_datapoints_w_marker_estimates = True,
        markers_use_advanced_removal=True,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.05, use_depth_images_if_provided=True),
        est3d_xyz_icp_config=ICPAlignmentConfig(do_alginment=False)
)

labeled_headset_data_2 = bind_headset_recording_to_scan(
        headset_data = HeadsetRecording.from_vrs_file(vrs_file_location),
        robot_data = robot_data_2
)

In [ ]:
lg = LineGenerator(line_cleanup_config=    
    MultiPassLineMergingConfig(passes=[
        LineMerging2dConfig(max_angle_diff = 2, max_midpoint_dist = 1/850, max_endpoint_dist = 0.02, min_line_length = 40/850, use_quick_merge = True),
        LineMerging2dConfig(max_angle_diff = 3, max_midpoint_dist = 2/850, max_endpoint_dist = 0.04, min_line_length = 40/850, use_quick_merge = True),
    ]),
    visualize_cleanup=True)
lg.get_lines(robot_env_2.robot_bgr_images[7])

### Comparison of different 3D reconstruction settings
This section does an ablation study, using the PnP LoMa predictor, of Depth and ICP Alignment and their influence on performance

In [ ]:
from shared.complete_robot_scan import CompleteRobotScan
from headset_localization import *

robot_data_folder_location = "../example_datasets/example_small_aruco1"
vrs_file_location = "../example_datasets/small_aruco1_sitting_20fps.vrs"


scans3d = []
names = []
predictors = []

robot_data = CompleteRobotScan.from_folder(robot_data_folder_location)
headset_recording = bind_headset_recording_to_scan(
        headset_data = HeadsetRecording.from_vrs_file(vrs_file_location),
        robot_data = robot_data
)

for use_depth, use_align in [(False, False), (False, True), (True, False), (True, True)]:
        print(f"creating: Depth: {use_depth}, Align: {use_align}")
        env3d = Scanned3dEnvironment.from_gathered_robot_data(
                robot_data = robot_data,
                number_of_sampled_datapoints=10,
                sample_datapoints_based_on_aruco_corectness = False,
                only_sample_robot_datapoints_w_marker_estimates = True,
                markers_use_advanced_removal=True,
                est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.05, use_depth_images_if_provided=use_depth),
                est3d_xyz_icp_config=ICPAlignmentConfig(do_alginment=use_align)
        )
        scans3d.append(env3d)
        
        predictors.append(PnPLocalizer(
                        cam2_intrinsic_mtx=headset_recording.intrinsic_cam_mtx,
                        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                                extract_and_match=ExtractAndMatchLoMa('LoMaB128')
                        ),
                        cam1_bgr_images=env3d.robot_bgr_images,
                        cam1_xyz_images=env3d.robot_xyz_images
                )
        )
        names.append(f"Depth: {use_depth}, Align: {use_align}")

import pandas as pd
import matplotlib.pyplot as plt

results = []

for name, predictor in zip(names, predictors):
    pod = PredictionOnDataset(predictor=predictor, headset_data=headset_recording)
    
    results.append({
        'Name': name,
        'Error [mm]': pod.avg_translational_error*1000,
        'Error [deg]': np.rad2deg(pod.avg_rotational_error),
        'Error [mm] ': pod.median_translational_error*1000,
        'Error [deg] ': np.rad2deg(pod.median_rotational_error)
    })
df_results = pd.DataFrame(results)

In [ ]:
fig, axes = plt.subplots(1,2, figsize = (12, 2.0))

NPredictors1DatasetGrader._plot_frontier_plot(
    df=df_results,
    xkey='Error [mm]',
    ykey='Error [deg]',
    title="Translational vs Rotational MAE",
    invert_y=False,
    name_key="Name",
    full_name_key="Name",
    ax=axes[0],
    adjust_texts=False,
    plot_names=False,
    plot_legend=True
)

NPredictors1DatasetGrader._plot_frontier_plot(
    df=df_results,
    xkey='Error [mm] ',
    ykey='Error [deg] ',
    title="Translational vs Rotational Median Errors",
    invert_y=False,
    name_key="Name",
    full_name_key="Name",
    ax=axes[1],
    adjust_texts=False,
    plot_names=False,
    plot_legend=False
)

plt.show()